# 02. EDA y entendimiento del dominio

**Fases del guía metodológica cubiertas: 4 (EDA y dominio)**

> Regla central aplicada: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.
> El test nunca influye en preprocessing, selección de variables, hiperparámetros o elección de modelo.



## 4.1 EDA del target

El target es el consumo de fin de semana `Walc` (escala 1-5, donde 1 = muy bajo y
5 = muy alto). Antes de binarizar, analizamos su **distribución completa** para entender
el desbalance y la forma de la curva: si la mayoría de alumnos se concentra en 1-2, el
problema binario tendrá ~35-40 % de positivos (consumo alto = 3-5). También comparamos
con `Dalc` (consumo en día laborable): la distribución de fin de semana suele estar más
dispersa, lo que indica que es una variable más discriminante. Generamos un gráfico de
barras para ambas variables y calculamos la prevalencia exacta del target binarizado.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np, pandas as pd

from src.data.load_data import load_student_dataset
from src.features.build_features import build_dataset_unique_students
mat, por, merged = load_student_dataset()
# Dataset de alumnos UNICOS (662): 1 fila por alumno, priorizando Matematicas
merge_cols = ["school","sex","age","address","famsize","Pstatus","Medu","Fedu","Mjob","Fjob","reason","nursery","internet"]
df = build_dataset_unique_students(mat, por, merge_cols)
print("EDA sobre", len(df), "alumnos unicos")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["Dalc"].value_counts().sort_index().plot.bar(ax=axes[0], title="Dalc (día laborable)")
df["Walc"].value_counts().sort_index().plot.bar(ax=axes[1], title="Walc (fin de semana)", color="coral")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "02_target_distribucion.png", dpi=120)
plt.show()

prevalencia_alto = (df["Walc"] >= 3).mean()
print(f"Prevalencia consumo alto (Walc>=3): {prevalencia_alto:.1%}")


EDA sobre 662 alumnos unicos


Prevalencia consumo alto (Walc>=3): 39.3%


C:\Users\sgml1\AppData\Local\Temp\ipykernel_17304\3878238190.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## 4.2 EDA de features

### 4.2.1 Estadísticos descriptivos

Calculamos los estadísticos principales (media, desviación, mínimos, cuartiles y máximos)
de todas las variables numéricas y ordinales. Esta tabla permite detectar valores
sospechosos (por ejemplo, `absences` con máximo 93 frente a una mediana de 4) y
comprender la escala de cada variable, lo que será útil para decidir si hace falta
escalado (no para árboles, sí para regresión logística) y para interpretar después los
coeficientes o importancias.


In [2]:

df.describe().T.round(2)


,count,mean,std,min,25%,50%,75%,max
age,662.0,16.81,1.27,15.0,16.0,17.0,18.0,22.0
Medu,662.0,2.49,1.13,0.0,2.0,2.0,4.0,4.0
Fedu,662.0,2.29,1.09,0.0,1.0,2.0,3.0,4.0
traveltime,662.0,1.56,0.74,1.0,1.0,1.0,2.0,4.0
studytime,662.0,1.93,0.83,1.0,1.0,2.0,2.0,4.0
failures,662.0,0.33,0.72,0.0,0.0,0.0,0.0,3.0
famrel,662.0,3.94,0.94,1.0,4.0,4.0,5.0,5.0
freetime,662.0,3.18,1.06,1.0,3.0,3.0,4.0,5.0
goout,662.0,3.17,1.16,1.0,2.0,3.0,4.0,5.0
Dalc,662.0,1.50,0.93,1.0,1.0,1.0,2.0,5.0



### 4.2.2 Distribuciones por clase del target

Para cada variable numérica/ordinal relevante, dibujamos un **histograma separado por
clase** del target (consumo alto vs bajo). Esto muestra visualmente qué variables
discriminan mejor: si las barras de las dos clases están muy separadas (p. ej. en `goout`
o `Dalc`), la variable será un predictor potente; si se solapan por completo (p. ej.
`Medu`), su poder predictivo será escaso. Guardamos la figura en `reports/figures/`.


In [3]:

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
num_cols = ["age", "Medu", "Fedu", "traveltime", "studytime", "failures", "goout", "absences"]
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(data=df, x=c, hue="Walc", ax=ax, palette="viridis", multiple="dodge", shrink=0.8)
    ax.set_title(c)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "02_hist_features.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_17304\3982958620.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 4.2.3 Matriz de correlaciones

La **matriz de correlación de Pearson** entre todas las variables numéricas/ordinales nos
advierte de la **redundancia**: variables con correlación > 0.7 (p. ej. `G1-G3`, que no
usaremos) transmiten información casi duplicada. También muestra la relación lineal con
el target (columna/fila de `Walc`). Es importante recordar que la correlación de Pearson
solo captura relaciones lineales: en el análisis de información mutua (fase 10) veremos
las relaciones no lineales.


In [4]:

# Correlaciones (numéricas/ordinales)
num = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
sns.heatmap(num.corr(), annot=False, cmap="coolwarm", center=0)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "02_correlaciones.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_17304\86644925.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 4.2.4 Relaciones lineales con el target (correlación punto-biserial)

Para cuantificar la fuerza de cada variable con el target binarizado usamos la
**correlación punto-biserial** (equivalente a Pearson cuando una variable es binaria).
Imprimimos el coeficiente `r` y el p-valor: un p-valor < 0.05 indica asociación
estadísticamente significativa. Esperamos que `goout` y `Dalc` tengan `r` positivo alto
y `famrel` o `studytime` negativo (factores protectores). Esta tabla orienta las
hipótesis de la sección 4.4 y el feature engineering de la fase 9.


In [5]:

from scipy.stats import pointbiserialr, chi2_contingency

y = (df["Walc"] >= 3).astype(int)
for c in ["goout", "Dalc", "age", "absences", "studytime", "failures", "famrel", "health", "freetime"]:
    r, p = pointbiserialr(df[c], y)
    print(f"{c:10s} r={r:+.3f}  p={p:.2e}")


goout      r=+0.352  p=8.94e-21
Dalc       r=+0.497  p=1.26e-42
age        r=+0.065  p=9.41e-02
absences   r=+0.139  p=3.48e-04
studytime  r=-0.169  p=1.25e-05
failures   r=+0.089  p=2.20e-02
famrel     r=-0.072  p=6.43e-02
health     r=+0.094  p=1.51e-02
freetime   r=+0.155  p=6.25e-05



### 4.2.5 Categóricas vs target

Para las variables categóricas dibujamos la **proporción de consumo alto dentro de cada
categoría** (barras apiladas normalizadas). Esto revela, por ejemplo, si los alumnos que
tienen relación de pareja (`romantic=yes`) presentan mayor tasa de consumo alto, o si el
sexo está asociado al consumo. Las categorías con diferencias grandes serán candidatas a
entrar con fuerza en el modelo.


In [6]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), ["sex", "school", "address", "famsize", "Pstatus", "romantic"]):
    ct = pd.crosstab(df[c], y, normalize="index")
    ct.plot.bar(ax=ax, stacked=True, color=["#4c72b0", "#c44e52"], legend=False)
    ax.set_title(f"{c}  (tasa alto = {ct[1].max():.0%})")
    ax.set_ylabel("")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "02_categoricas_target.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_17304\3568674810.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## 4.3 EDA específico (tabular)

No aplican secciones de visión/NLP/series temporales. Para tabular:

- **Cardinalidad de categóricas**: todas <= 5 categorías (ver `01`).
- **Categorías raras**: `Mjob`/`Fjob` con categorías minoritarias (teacher/health) — los
  árboles lo manejan; OneHot con `handle_unknown="ignore"`.
- **Nulos**: 0 nulos en el dataset crudo.
- **Redundancia**: `Medu`-`Fedu` correlacionadas (~0.6); `Dalc`-`Walc` (0.65).

## 4.4 Hipótesis

1. `goout` (salir con amigos) y `Dalc` (consumo entre semana) son los predictores más fuertes.
2. `age` y `romantic` aumentan el riesgo; `famrel`, `health` y `studytime` lo reducen.
3. El apoyo familiar (`schoolsup`/`famsup`) modera el riesgo.
4. Las calificaciones G1-G3 correlacionan con el consumo, pero **son post-evento**: usar
   su información sería fuga.
5. Un baseline de regla simple (goout>=4 y Dalc>=3) es razonable como referencia.
6. Split estratificado aleatorio (no hay repetición de alumnos ni orden temporal).
